# Data quality checks tutorial: Legal Entities

This tutorial shows you how to create and run a DQ check on a set of legal entities.

## Section 1: Create and test a check definition
In section 1, the notebook follows the steps explained in [How do I create and run a data quality check?](https://support.lusid.com/docs/how-do-i-create-and-run-a-data-quality-check).
1. Set up LUSID legal entities and property definitions that we plan to run DQ checks on.
2. Decorate legal entities with properties.
3. Create a check definition that defines which data to run the checks on and any limits for each rule.
4. Add rules to the check definition.
5. Run the check on some data.

## Section 2: Set up a DQ check workflow
In section 2, the notebook follows the steps explained in [How do I set up a data quality check workflow?](https://support.lusid.com/docs/how-do-i-set-up-a-data-quality-check-workflow).
1. Create exception task definition to handle DQ check results (breaches).
2. Create DQ check task definition, the main task we'll run whenever we want to perform a DQ check.
3. Kick off a task test run.
4. Inspect the results.


## Setup
Build the LUSID and Workflow APIs and create some test data to run a DQ check on

In [ ]:
# import the latest SDK version
#!pip3 install -U finbourne-sdk

import os
import time
from pprint import pprint

import pandas as pd

import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as lm
import finbourne.sdk.services.workflow as lw
import finbourne.sdk.services.workflow.models as lwm

from finbourne.sdk.extensions import SyncApiClientFactory
from finbourne.sdk.exceptions import ApiException

secrets_path = os.getenv("FBN_SECRETS_PATH")
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

api_factory = SyncApiClientFactory(secrets_path=secrets_path, app_name="LusidJupyterNotebook")

pd.DataFrame(api_factory.build(lu.ApplicationMetadataApi).get_lusid_versions().to_dict())

In [ ]:
# Helper functions for compact list construction
def fields(*items):
    return [lwm.TaskFieldDefinition(name=n, type=t, display_name=d) for n, t, d in items]

def states(*names):
    return [lwm.TaskStateDefinition(name=n, display_name=n, description=n) for n in names]

def triggers(*names):
    return [lwm.TransitionTriggerDefinition(name=n, trigger=lwm.TriggerSchema(type="External")) for n in names]

def transitions(*items):
    return [lwm.TaskTransitionDefinition(from_state=f, to_state=t, trigger=tr, **kw) for f, t, tr, *rest in items for kw in [rest[0] if rest else {}]]

def map_from(field):
    return lwm.FieldMapping(map_from=field, set_to=None)

In [ ]:
# Specify a unique scope and code to segregate data in this example
module_scope = "Finbourne-Examples"
module_code = "DQ-Check-le"
print(f"'{module_scope}\\{module_code}' scope and code created.")

In [ ]:
checkDefinitions_api = api_factory.build(lu.CheckDefinitionsApi)
task_definitions_api = api_factory.build(lw.TaskDefinitionsApi)
tasks_api = api_factory.build(lw.TasksApi)

## Section 1: Create and test a check definition
### Step 1: Set up legal entities in LUSID

#### Create property definitions

Let's imagine we want to check that all legal entities with the `LegalEntity/<scope>/RefreshData` property set to `True` have the `LegalEntity/<scope>/Country` property populated. We'll first need to create those property definitions:

In [ ]:
# Obtain the Property Definitions API
property_definitions_api = api_factory.build(lu.PropertyDefinitionsApi)

try:
    country_le_prop_definition = property_definitions_api.create_property_definition(
        create_property_definition_request=lm.CreatePropertyDefinitionRequest(
            domain="LegalEntity",
            scope=f"{module_scope}{module_code}",
            code="Country",
            display_name="Country",
            property_description="Tags the country a legal entity is associated with, for DQ filtering",
            data_type_id=lm.ResourceId(scope="system", code="string"),
            life_time="Perpetual",
            constraint_style="Property"
        )
    )
    pprint(country_le_prop_definition)
except ApiException as e:
    if e.status == 400 and "PropertyAlreadyExists" in str(e.body):
        print("Property 'Country' already exists, skipping creation.")
    else:
        raise

In [ ]:
try:
    refreshData_le_prop_definition = property_definitions_api.create_property_definition(
        create_property_definition_request=lm.CreatePropertyDefinitionRequest(
            domain="LegalEntity",
            scope=f"{module_scope}{module_code}",
            code="RefreshData",
            display_name="RefreshData",
            property_description="Tags data that should be included in integration runs",
            data_type_id=lm.ResourceId(scope="system", code="string"),
            life_time="Perpetual",
            constraint_style="Property"
        )
    )
    pprint(refreshData_le_prop_definition)
except ApiException as e:
    if e.status == 400 and "PropertyAlreadyExists" in str(e.body):
        print("Property 'RefreshData' already exists, skipping creation.")
    else:
        raise

#### Create legal entities

We'll create two legal entities using the `LegalEntity/InternationalBanks/BankId` identifier scheme: one that will end up decorated with both `Country` and `RefreshData` (so it satisfies the check definition's `ruleSetFilter` and passes the `Country`-exists rule we'll add later), and one with `RefreshData` only, missing `Country`, so it breaches that rule.

In [ ]:
legal_entities_api = api_factory.build(lu.LegalEntitiesApi)

le_id_type_scope = "InternationalBanks"
le_id_type_code = "BankId"
acme_bank_id = "AcmeInc"
no_property_bank_id = f"{module_code}-noProperty"


In [ ]:
le_bank_id_property_definition = lm.CreatePropertyDefinitionRequest(
    domain="LegalEntity",
    scope=le_id_type_scope,
    code=le_id_type_code,
    display_name="Bank ID",
    property_description="Identifier used to uniquely identify a legal entity within the InternationalBanks scheme",
    data_type_id=lm.ResourceId(scope="system", code="string"),
    life_time="Perpetual",
    constraint_style="Identifier"
)

try:
    property_definitions_api.create_property_definition(
        create_property_definition_request=le_bank_id_property_definition
    )
    print(f"Property definition 'LegalEntity/{le_id_type_scope}/{le_id_type_code}' created.")
except ApiException as e:
    if e.status == 400 and "PropertyAlreadyExists" in str(e.body):
        print(f"Property definition 'LegalEntity/{le_id_type_scope}/{le_id_type_code}' already exists, skipping creation.")
    else:
        raise

In [ ]:
try:
    upsert_legal_entities_response = legal_entities_api.upsert_legal_entities(
        success_mode="Partial",
        request_body={
            "acme-bank-inc": lm.UpsertLegalEntityRequest(
                identifiers={
                    f"LegalEntity/{le_id_type_scope}/{le_id_type_code}": lm.ModelProperty(
                        key=f"LegalEntity/{le_id_type_scope}/{le_id_type_code}",
                        value=lm.PropertyValue(label_value=acme_bank_id)
                    )
                },
                display_name="Acme Bank Inc",
                description="A US bank"
            ),
            "no-property-bank": lm.UpsertLegalEntityRequest(
                identifiers={
                    f"LegalEntity/{le_id_type_scope}/{le_id_type_code}": lm.ModelProperty(
                        key=f"LegalEntity/{le_id_type_scope}/{le_id_type_code}",
                        value=lm.PropertyValue(label_value=no_property_bank_id)
                    )
                },
                display_name="No Property Bank",
                description="A legal entity created to test DQ checks against the LegalEntity entity type"
            )
        }
    )
    print(f"Successfully upserted {len(upsert_legal_entities_response.values)} legal entities.")
    display(upsert_legal_entities_response)
except ApiException as e:
    print(f"Error upserting legal entities: {e}")

### Step 2: Decorate legal entities with properties

Set `Country` and `RefreshData` on Acme Bank Inc, and `RefreshData` only on the no-property bank.

In [ ]:
le_country_property_key = f"LegalEntity/{module_scope}{module_code}/Country"
le_refresh_data_property_key = f"LegalEntity/{module_scope}{module_code}/RefreshData"

try:
    set_acme_properties_response = legal_entities_api.set_legal_entity_properties(
        id_type_scope=le_id_type_scope,
        id_type_code=le_id_type_code,
        code=acme_bank_id,
        set_legal_entity_properties_request=lm.SetLegalEntityPropertiesRequest(
            properties={
                le_country_property_key: lm.ModelProperty(
                    key=le_country_property_key,
                    value=lm.PropertyValue(label_value="US")
                ),
                le_refresh_data_property_key: lm.ModelProperty(
                    key=le_refresh_data_property_key,
                    value=lm.PropertyValue(label_value="True")
                )
            }
        )
    )
    print(f"Successfully set properties on legal entity '{acme_bank_id}'.")
    display(set_acme_properties_response)
except ApiException as e:
    print(f"Error setting properties on '{acme_bank_id}': {e}")

In [ ]:
try:
    set_no_property_bank_response = legal_entities_api.set_legal_entity_properties(
        id_type_scope=le_id_type_scope,
        id_type_code=le_id_type_code,
        code=no_property_bank_id,
        set_legal_entity_properties_request=lm.SetLegalEntityPropertiesRequest(
            properties={
                le_refresh_data_property_key: lm.ModelProperty(
                    key=le_refresh_data_property_key,
                    value=lm.PropertyValue(label_value="True")
                )
            }
        )
    )
    print(f"Successfully set properties on legal entity '{no_property_bank_id}'.")
    display(set_no_property_bank_response)
except ApiException as e:
    print(f"Error setting properties on '{no_property_bank_id}': {e}")

### Step 3: Create a check definition that contains empty rulesets


In [ ]:
legal_entity_rule_set_key = "legal-entity-properties-checks"
le_rule_sets = [lm.UpdateCheckDefinitionRuleSet(
    rule_set_key = legal_entity_rule_set_key,
    display_name = "Legal entity properties checks ruleset",
    description = "A set of rules to apply to legal entities assigned for data refreshes to check for appropriate properties.",
    rule_set_filter = f"Properties[LegalEntity/{module_scope}{module_code}/RefreshData] exists"
)]
try:
    create_le_check_definition_request = lm.CreateCheckDefinitionRequest(
        id = lm.ResourceId(
            scope = module_scope,
            code = f"{module_code}-legal-entity-properties"
        ),
        display_name = "Legal entities check",
        description = "A check definition to validate legal entities are populated with the correct properties",
        dataset_schema = lm.CheckDefinitionDatasetSchema(
            type = "LusidEntity",
            entity_type = "LegalEntity"
        ),
        rule_sets = le_rule_sets
    )

    create_le_check_definition_response = checkDefinitions_api.create_check_definition(create_check_definition_request=create_le_check_definition_request)
    pprint(create_le_check_definition_response)
except ApiException as e:
    if e.status == 400 and "EntityWithIdAlreadyExists" in str(e.body):
        print(f"Check definition '{module_code}-legal-entity-properties' already exists in scope '{module_scope}', skipping creation.")
    else:
        raise

### Step 4: Add a rule to the check definition

In [ ]:
# We want to create the following rule:
# "Check legal entities have the property LegalEntity/<scope>/Country"
le_check_definition_rule = lm.CheckDefinitionRule(
    rule_key = "country-exists",
    display_name = "Country exists check",
    description = f"Checks whether a legal entity is decorated with the LegalEntity/{module_scope}{module_code}/Country property",
    rule_formula = f"properties[LegalEntity/{module_scope}{module_code}/Country] exists",
    severity = 1
)
try:
    upsert_le_data_quality_rule = [lm.UpsertDataQualityRule(
        rule_set_key=legal_entity_rule_set_key,
        rule=le_check_definition_rule
    )]

    upsert_le_rule_response = checkDefinitions_api.upsert_rules(scope=module_scope, code=f"{module_code}-legal-entity-properties", upsert_data_quality_rule=upsert_le_data_quality_rule)

    print(f"Successfully upserted rule '{le_check_definition_rule.rule_key}' to ruleset '{legal_entity_rule_set_key}'.")
    pprint(upsert_le_rule_response)
except ApiException as e:
    print(f"Error creating ruleset: {e}")

### Step 5: Run the check on some data

Running the check, we can see that LUSID outputs a result for each legal entity with the `RefreshData` property set to `True` that is missing the `LegalEntity/<scope>/Country` property.


In [ ]:
try:
    run_le_check_request = lm.RunCheckRequest(
        lusid_entity_dataset = lm.LusidEntityDataset(
            selector_attribute = f"Properties[LegalEntity/{module_scope}{module_code}/RefreshData]",
            selector_value = "True",
            return_identifier_key = f"LegalEntity/{le_id_type_scope}/{le_id_type_code}"
        ),
        limit_individual_breaches_per_rule = 100
    )

    run_le_check_response = checkDefinitions_api.run_check_definition(scope=module_scope, code=f"{module_code}-legal-entity-properties", run_check_request=run_le_check_request)
    le_results = run_le_check_response.data_quality_check_results
    le_total = len(le_results)
    print(f"Check complete: {le_total} results (breaches)")
except ApiException as e:
    print(f"Error running check: {e}")

## Section 2: Set up a DQ check workflow
### Step 1: Create exception task definition

We need to create this task definition first so we can reference it in the main DQ check task definition.

LUSID will create one exception task for each DQ check result (breach).

In [ ]:
legal_entity_exception_task_def = lwm.CreateTaskDefinitionRequest(
    id=lwm.ResourceId(scope=module_scope, code=f"{module_code}-legal-entity-exception"),
    display_name="DQ Legal Entity Exception",
    description="An exception returned by a data quality check.",
    states=states("Pending", "InProgress", "Resolved"),
    field_schema=fields(
        ("checkDefinitionScope",       "String",   "CheckDef Scope"),
        ("checkDefinitionCode",        "String",   "CheckDef Code"),
        ("checkDefinitionDisplayName", "String",   "CheckDef Name"),
        ("checkRunAsAt",               "DateTime", "Run AsAt"),
        ("resultType",                 "String",   "Result Type"),
        ("rulesetKey",                 "String",   "Ruleset Key"),
        ("rulesetDisplayName",         "String",   "Ruleset Name"),
        ("ruleKey",                    "String",   "Rule Key"),
        ("ruleDisplayName",            "String",   "Rule Name"),
        ("ruleDescription",            "String",   "Rule Description"),
        ("ruleFormula",                "String",   "Rule Formula"),
        ("severity",                   "Decimal",  "Severity"),
        ("resultId",                   "String",   "Result Tracking ID"),
        ("entityType",                 "String",   "Entity Type"),
        ("legalEntityAsAt",            "DateTime", "Legal Entity As At"),
        ("legalEntityEffectiveAt",     "DateTime", "Legal Entity Effective At"),
        ("identifierType",             "String",   "Identifier Type"),
        ("identifierValue",            "String",   "Identifier Value"),
        ("legalEntityName",            "String",   "Legal Entity Name"),
        ("entityUniqueId",             "String",   "Entity Unique ID"),
        ("countRuleBreaches",          "Decimal",  "Number of Breaches"),
        ("errorDetail",                "String",   "Error Message"),
    ),
    initial_state=lwm.InitialState(name="Pending", required_fields=[]),
    triggers=triggers("start", "resolve"),
    actions=[
        lwm.ActionDefinition(
            name="resolve-parent",
            action_details=lwm.ActionDetails(
                lwm.TriggerParentTaskAction(type="TriggerParentTask", trigger="resolve")
            )
        )
    ],
    transitions=transitions(
        ("Pending",    "InProgress", "start"),
        ("InProgress", "Resolved",   "resolve", {"action": "resolve-parent"}),
    )
)
try:
    legal_entity_exception_response = task_definitions_api.create_task_definition(
        create_task_definition_request=legal_entity_exception_task_def
    )
    print(f"Task definition created successfully. Scope: {legal_entity_exception_response.id.scope}, Code: {legal_entity_exception_response.id.code}")
except ApiException as e:
    print(f"Error creating exception task definition: {e}")

### Step 2: Create DQ check task definition

This is the parent task definition that we can run each time we want to perform a DQ check on some data.

We pass in the scope and code of our check definition to LUSID's built-in DQ check worker `worker_id=lwm.ResourceId(scope="default", code="LusidEntityDataQuality")`. By default, the worker can kick off runs of the specified DQ check and turn each result into its own exception task for resolution.

In [ ]:
legal_entity_child_task_fields = {k: map_from(v) for k, v in [
    ("checkDefinitionScope",       "checkDefinitionScope"),
    ("checkDefinitionCode",        "checkDefinitionCode"),
    ("checkDefinitionDisplayName", "checkDefinitionDisplayName"),
    ("checkRunAsAt",               "checkRunAsAt"),
    ("resultType",                 "resultType"),
    ("rulesetKey",                 "rulesetKey"),
    ("rulesetDisplayName",         "rulesetDisplayName"),
    ("ruleKey",                    "ruleKey"),
    ("ruleDisplayName",            "ruleDisplayName"),
    ("ruleDescription",            "ruleDescription"),
    ("ruleFormula",                "ruleFormula"),
    ("resultId",                   "resultId"),
    ("entityType",                 "lusidEntityType"),
    ("legalEntityAsAt",            "lusidEntityAsAt"),
    ("legalEntityEffectiveAt",     "lusidEntityEffectiveAt"),
    ("identifierType",             "lusidEntityIdentifierKey"),
    ("identifierValue",            "lusidEntityIdentifierValue"),
    ("legalEntityName",            "lusidEntityDisplayName"),
    ("entityUniqueId",             "lusidEntityUniqueId"),
    ("severity",                   "severity"),
    ("countRuleBreaches",          "countRuleBreaches"),
    ("errorDetail",                "errorDetail"),
]}

legal_entity_dq_check_task_def = lwm.CreateTaskDefinitionRequest(
    id=lwm.ResourceId(scope=module_scope, code=f"{module_code}-CheckLegalEntities"),
    display_name="DQ Check Legal Entities",
    description="Runs data quality checks for legal entities.",
    states=states("Pending", "InProgress", "ExceptionManagement", "Complete", "Error"),
    field_schema=fields(
        ("checkDefinitionScope", "String",  "CD Scope"),
        ("checkDefinitionCode",  "String",  "CD Code"),
        ("selectorAttribute",    "String",  "Selector Attribute"),
        ("selectorValue",        "String",  "Selector Value"),
        ("preferredIdentifier",  "String",  "Preferred Identifier"),
        ("ruleBreachLimit",      "Decimal", "Rule Breach Limit"),
    ),
    initial_state=lwm.InitialState(name="Pending", required_fields=[]),
    triggers=triggers("start", "exceptions", "no_exceptions", "resolve", "error"),
    actions=[
        lwm.ActionDefinition(
            name="run-checks",
            action_details=lwm.ActionDetails(
                lwm.RunWorkerAction(
                    type="RunWorker",
                    worker_id=lwm.ResourceId(scope="default", code="LusidEntityDataQuality"),
                    worker_parameters={
                        "checkDefinitionScope":           map_from("checkDefinitionScope"),
                        "checkDefinitionCode":            map_from("checkDefinitionCode"),
                        "selectorAttribute":              map_from("selectorAttribute"),
                        "selectorValue":                  map_from("selectorValue"),
                        "returnIdentifierKey":            map_from("preferredIdentifier"),
                        "limitIndividualBreachesPerRule": map_from("ruleBreachLimit"),
                    },
                    worker_status_triggers=lwm.WorkerStatusTriggers(
                        started=None,
                        completed_with_results="exceptions",
                        completed_no_results="no_exceptions",
                        failed_to_start="error",
                        failed_to_complete="error"
                    ),
                    child_task_configurations=[
                        lwm.ResultantChildTaskConfiguration(
                            task_definition_id=lwm.ResourceId(scope=module_scope, code=f"{module_code}-legal-entity-exception"),
                            map_stacking_key_from="resultId",
                            child_task_fields=legal_entity_child_task_fields,
                            result_matching_pattern=None,
                            initial_trigger=None
                        )
                    ]
                )
            )
        )
    ],
    transitions=transitions(
        ("Pending",             "InProgress",          "start",         {"action": "run-checks"}),
        ("InProgress",          "Complete",            "no_exceptions"),
        ("InProgress",          "ExceptionManagement", "exceptions"),
        ("ExceptionManagement", "Complete",            "resolve",       {"guard": "childTasks all (state eq 'Resolved')"}),
        ("InProgress",          "Error",               "error"),
    )
)

try:
    legal_entity_dq_check_response = task_definitions_api.create_task_definition(
        create_task_definition_request=legal_entity_dq_check_task_def
    )
    print(f"Task definition created successfully. Scope: {legal_entity_dq_check_response.id.scope}, Code: {legal_entity_dq_check_response.id.code}")
except ApiException as e:
    print(f"Error creating DQ check task definition: {e}")

### Step 3: Kick off a task test run

In [ ]:
create_legal_entity_task_request = lwm.CreateTaskRequest(
    task_definition_id=lwm.ResourceId(scope=module_scope, code=f"{module_code}-CheckLegalEntities"),
    correlation_ids=[],
    fields=[
        lwm.TaskInstanceField(
            name="checkDefinitionScope",
            value=module_scope
        ),
        lwm.TaskInstanceField(
            name="checkDefinitionCode",
            value=f"{module_code}-legal-entity-properties"
        ),
        lwm.TaskInstanceField(
            name="selectorAttribute",
            value=f"Properties[LegalEntity/{module_scope}{module_code}/RefreshData]"
        ),
        lwm.TaskInstanceField(
            name="selectorValue",
            value="True"
        ),
        lwm.TaskInstanceField(
            name="preferredIdentifier",
            value=f"LegalEntity/{le_id_type_scope}/{le_id_type_code}"
        ),
        lwm.TaskInstanceField(
            name="ruleBreachLimit",
            value="100"
        ),
    ]
)

try:
    create_legal_entity_task_response = tasks_api.create_task(
        create_task_request=create_legal_entity_task_request,
        trigger="start"
    )
    legal_entity_task_id = create_legal_entity_task_response.id
    print(f"Task created successfully. ID: {legal_entity_task_id}")
except ApiException as e:
    print(f"Error creating task: {e}")

### Step 4: Inspect the results

Polls for child tasks every 10 seconds, up to a maximum of 2 minutes, stopping as soon as results are found.

In [ ]:
def display_legal_entity_exception_tasks(tasks):
    rows = []
    for task in tasks:
        field_dict = {f.name: f.value for f in task.fields}
        rows.append({
            "Task ID":          task.id,
            "State":            task.state,
            "Legal Entity Name": field_dict.get("legalEntityName"),
            "Identifier":       f"{field_dict.get('identifierType')}: {field_dict.get('identifierValue')}",
            "Rule":             field_dict.get("ruleDisplayName"),
            "Rule Formula":     field_dict.get("ruleFormula"),
            "Result Type":      field_dict.get("resultType"),
            "Severity":         field_dict.get("severity"),
            "Ruleset":          field_dict.get("rulesetDisplayName"),
            "Check Run At":     field_dict.get("checkRunAsAt"),
        })
    display(pd.DataFrame(rows))

max_attempts = 12
wait_seconds = 10
all_legal_entity_exception_tasks = []

for attempt in range(max_attempts):
    print(f"Checking for child tasks (attempt {attempt + 1}/{max_attempts})...")
    try:
        response = tasks_api.list_tasks(
            filter=f"ultimateParentTask.id eq '{legal_entity_task_id}'"
        )
        found = [t for t in response.values if t.parent_task is not None]

        while response.next_page:
            response = tasks_api.list_tasks(
                filter=f"ultimateParentTask.id eq '{legal_entity_task_id}'",
                page=response.next_page
            )
            found.extend([t for t in response.values if t.parent_task is not None])

        if found:
            all_legal_entity_exception_tasks = found
            print(f"Found {len(all_legal_entity_exception_tasks)} exception tasks.")
            break
    except ApiException as e:
        print(f"Error retrieving child tasks: {e}")
        break

    time.sleep(wait_seconds)
else:
    print("No exception tasks found after maximum wait time.")

if all_legal_entity_exception_tasks:
    display_legal_entity_exception_tasks(all_legal_entity_exception_tasks)

## Next steps

You can then resolve breaks and manage your tasks via the LUSID web app. [Read more.](https://support.lusid.com/docs/how-do-i-set-up-a-data-quality-check-workflow)


## Teardown

In [ ]:
try:
    task_definitions_api.delete_task_definition(scope=module_scope, code=f"{module_code}-CheckLegalEntities")
    print(f"Deleted task definition '{module_code}-CheckLegalEntities'.")
except ApiException as e:
    if e.status == 404:
        print(f"Task definition '{module_code}-CheckLegalEntities' did not exist, skipping.")
    else:
        print(f"Error deleting task definition '{module_code}-CheckLegalEntities': {e}")

try:
    task_definitions_api.delete_task_definition(scope=module_scope, code=f"{module_code}-legal-entity-exception")
    print(f"Deleted task definition '{module_code}-legal-entity-exception'.")
except ApiException as e:
    if e.status == 404:
        print(f"Task definition '{module_code}-legal-entity-exception' did not exist, skipping.")
    else:
        print(f"Error deleting task definition '{module_code}-legal-entity-exception': {e}")

# Check definition
try:
    checkDefinitions_api.delete_check_definition(scope=module_scope, code=f"{module_code}-legal-entity-properties")
    print(f"Deleted check definition '{module_code}-legal-entity-properties'.")
except ApiException as e:
    if e.status == 404:
        print(f"Check definition '{module_code}-legal-entity-properties' did not exist, skipping.")
    else:
        print(f"Error deleting check definition '{module_code}-legal-entity-properties': {e}")

# Legal entities
try:
    legal_entities_api.delete_legal_entity(id_type_scope=le_id_type_scope, id_type_code=le_id_type_code, code=acme_bank_id)
    print(f"Deleted legal entity '{acme_bank_id}'.")
except ApiException as e:
    if e.status == 404:
        print(f"Legal entity '{acme_bank_id}' did not exist, skipping.")
    else:
        print(f"Error deleting legal entity '{acme_bank_id}': {e}")

try:
    legal_entities_api.delete_legal_entity(id_type_scope=le_id_type_scope, id_type_code=le_id_type_code, code=no_property_bank_id)
    print(f"Deleted legal entity '{no_property_bank_id}'.")
except ApiException as e:
    if e.status == 404:
        print(f"Legal entity '{no_property_bank_id}' did not exist, skipping.")
    else:
        print(f"Error deleting legal entity '{no_property_bank_id}': {e}")

try:
    property_definitions_api.delete_property_definition(domain="LegalEntity", scope=le_id_type_scope, code=le_id_type_code)
    print(f"Deleted property definition 'LegalEntity/{le_id_type_scope}/{le_id_type_code}'.")
except ApiException as e:
    if e.status == 404:
        print(f"Property definition 'LegalEntity/{le_id_type_scope}/{le_id_type_code}' did not exist, skipping.")
    else:
        print(f"Error deleting property definition 'LegalEntity/{le_id_type_scope}/{le_id_type_code}': {e}")        

# Property definitions
try:
    property_definitions_api.delete_property_definition(domain="LegalEntity", scope=f"{module_scope}{module_code}", code="Country")
    print(f"Deleted property definition 'LegalEntity/{module_scope}{module_code}/Country'.")
except ApiException as e:
    if e.status == 404:
        print(f"Property definition 'LegalEntity/{module_scope}{module_code}/Country' did not exist, skipping.")
    else:
        print(f"Error deleting property definition 'LegalEntity/{module_scope}{module_code}/Country': {e}")

try:
    property_definitions_api.delete_property_definition(domain="LegalEntity", scope=f"{module_scope}{module_code}", code="RefreshData")
    print(f"Deleted property definition 'LegalEntity/{module_scope}{module_code}/RefreshData'.")
except ApiException as e:
    if e.status == 404:
        print(f"Property definition 'LegalEntity/{module_scope}{module_code}/RefreshData' did not exist, skipping.")
    else:
        print(f"Error deleting property definition 'LegalEntity/{module_scope}{module_code}/RefreshData': {e}")